# Exercise 4: Creating advanced research agent
Companion notebook for the **Exercise 4: Creating advanced research agent** section of Chapter 4 -- the capstone example. It combines everything from this chapter: a Supervisor-Worker design, `Send`-based dynamic spawning, a search subgraph, node caching, and checkpointing with time travel.

> Add `OPENAI_API_KEY` / `TAVILY_API_KEY` to your `.env` for real summaries and web search. The agent still runs end-to-end without them -- it just records the gap and moves on, the same way `search_web` and `summarize_results` do throughout this chapter.

## Creation of State
Two states: `ResearchState` (the main workflow state, extended here with `fetching_initiated` to track whether fetch workers have been dispatched) and `FetchState` (the shape of state each individual URL-fetch worker deals with).

In [1]:
import os
import operator
from typing import Annotated
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import requests

from langgraph.graph import StateGraph, START, END
from langgraph.types import Command, Send, CachePolicy
from langgraph.cache.memory import InMemoryCache
from langgraph.checkpoint.memory import MemorySaver
from langchain_community.retrievers import WikipediaRetriever
from langchain_openai import ChatOpenAI
from tavily import TavilyClient

load_dotenv()

# Falls back to a dummy key so the client can be constructed even when OPENAI_API_KEY
# is not set; the guard inside summarize_results() stops it from ever being called.
llm = ChatOpenAI(model_name="gpt-4o", temperature=0.0, api_key=os.getenv("OPENAI_API_KEY", "sk-not-set"))
wiki = WikipediaRetriever()

C:\Users\himan\AppData\Local\Temp\ipykernel_34476\4039779792.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import WikipediaRetriever


In [2]:
def keep_non_empty(a: str, b: str) -> str:
    """Reducer that keeps the non-empty string value."""
    return a if a else b


def add_unique(a: list[str], b: list[str]) -> list[str]:
    """List reducer that concatenates like operator.add, but also dedupes -- needed because
    Wikipedia and web search can both surface the same URL."""
    return list(dict.fromkeys(a + b))


class ResearchState(BaseModel):
    """Main state for the research workflow with reducers for parallel execution."""

    query: Annotated[str, keep_non_empty] = Field(default="", description="The research question")
    candidate_urls: Annotated[list[str], add_unique] = Field(default_factory=list, description="Source URLs, merged + deduped")
    page_texts: Annotated[list[str], operator.add] = Field(default_factory=list, description="Fetched page content, merged")
    draft_summary: Annotated[str, keep_non_empty] = Field(default="", description="Preliminary summary")
    final_report: Annotated[str, keep_non_empty] = Field(default="", description="The finalized research report")
    errors: Annotated[list[str], operator.add] = Field(default_factory=list, description="Error messages, merged")
    fetching_initiated: bool = Field(default=False, description="Flag to track if fetch workers have been dispatched")

In [3]:
class FetchState(BaseModel):
    """State for individual URL fetch workers."""

    url: str = Field(description="Single URL to fetch")
    content: str = Field(default="", description="Fetched content from the URL")
    error: str = Field(default="", description="Error message if fetch fails")

## Creation of Nodes
Nine nodes: `supervisor` (coordinator), `dispatcher` (spawns fetch workers), `prepare_query`, `search_wikipedia`, `search_web`, `fetch_url`, `merge_results`, `summarize_results`, `error_handler`.

In [4]:
def prepare_query(state: ResearchState) -> dict:
    """Worker node: cleans and prepares the query."""
    cleaned = state.query.strip().lower() if state.query else ""
    return {"query": cleaned}

In [5]:
def search_wikipedia(state: ResearchState) -> dict:
    """Worker node: searches Wikipedia for relevant content."""
    query = state.query
    if not query:
        return {"candidate_urls": [], "page_texts": []}

    urls, texts = [], []
    try:
        docs = wiki.invoke(query)
        for doc in docs[:3]:  # limit to 3 documents
            url = doc.metadata.get("url", "")
            if not url or not url.startswith("http"):
                title = doc.metadata.get("title", "").replace(" ", "_")
                url = f"https://en.wikipedia.org/wiki/{title}"
            urls.append(url)
            texts.append(doc.page_content[:500])  # first 500 chars
    except Exception as e:
        return {"candidate_urls": [], "page_texts": [], "errors": [f"Wikipedia error: {e}"]}

    return {"candidate_urls": urls, "page_texts": texts}

In [6]:
def search_web(state: ResearchState) -> dict:
    """Worker node: searches the web using Tavily."""
    query = state.query
    if not query:
        return {"candidate_urls": []}

    web_urls, errors = [], []
    try:
        tavily_api_key = os.getenv("TAVILY_API_KEY")
        if not tavily_api_key:
            raise ValueError("TAVILY_API_KEY not set")
        tavily_client = TavilyClient(api_key=tavily_api_key)
        response = tavily_client.search(query=query, max_results=3)
        web_urls = [result["url"] for result in response.get("results", [])]
    except Exception as e:
        errors.append(f"Tavily error: {e}")

    return {"candidate_urls": web_urls, "errors": errors}

In [7]:
def fetch_url(state) -> dict:
    """Worker node: fetches content from a single URL (spawned via Send, one per URL)."""
    url = state.url if isinstance(state, FetchState) else state.get("url", "")
    try:
        headers = {"User-Agent": "Mozilla/5.0"}  # avoids some sites' 403 on bare requests
        response = requests.get(url, timeout=10, headers=headers)
        response.raise_for_status()
        return {"page_texts": [response.text[:5000]]}
    except Exception as e:
        return {"errors": [f"Error fetching {url}: {e}"]}

In [8]:
def merge_results(state: ResearchState) -> dict:
    """Aggregation node: both parallel searches already merged into candidate_urls via
    the (deduping) reducer. This node is just the single point the subgraph exits from."""
    print(f"  merged {len(state.candidate_urls)} unique source(s)")
    return {}

In [9]:
def summarize_results(state: ResearchState) -> dict:
    """Worker node: summarizes the gathered content."""
    combined = "\n\n".join(state.page_texts)
    if not combined or len(combined.strip()) < 10:
        return {"final_report": "No content found to summarize."}

    if not os.getenv("OPENAI_API_KEY"):
        return {"final_report": "[Skipped LLM call: set OPENAI_API_KEY to generate a real summary]"}

    try:
        prompt = f"Summarize the following research content concisely:\n\n{combined[:2000]}"
        response = llm.invoke(prompt)
        return {"final_report": response.content}
    except Exception as e:
        return {"final_report": f"Error generating summary: {e}"}

In [10]:
def error_handler(state: ResearchState) -> dict:
    """Worker node: handles errors and ends the workflow."""
    for error in state.errors:
        print(f"  - {error}")
    return {"final_report": f"Workflow completed with {len(state.errors)} error(s). Check errors list."}

**Supervisor node** -- the coordinator. It never does the work itself; it only reads state and returns a `Command` telling the graph what to do next.

In [11]:
def supervisor(state: ResearchState):
    """Supervisor node: coordinates the workflow using Command for runtime control."""
    query = state.query
    errors = state.errors
    urls = state.candidate_urls
    page_texts = state.page_texts
    fetching_initiated = state.fetching_initiated

    # 1. If fetch workers already ran and produced content, summarize it.
    if fetching_initiated and page_texts:
        print(f"[SUPERVISOR] {len(page_texts)} page texts collected - routing to summarize_results")
        return Command(goto="summarize_results")

    # 2. Errors with no usable content -> hand off to the error handler.
    if errors and not page_texts:
        print("[SUPERVISOR] Errors detected with no content - routing to error_handler")
        return Command(goto="error_handler")

    # 3. URLs found but not fetched yet -> dispatch fetch workers.
    if urls and not fetching_initiated:
        print(f"[SUPERVISOR] {len(urls)} URLs found - routing to dispatcher")
        return Command(goto="dispatcher")

    # 4. Already fetching -> wait for more fetch_url workers to report back.
    if fetching_initiated:
        print(f"[SUPERVISOR] Fetch in progress, {len(page_texts)} texts so far...")
        return {}

    # 5. Query not prepared yet.
    if not query:
        print("[SUPERVISOR] Preparing query")
        return Command(goto="prepare_query")

    # 6. No URLs yet -> run the search subgraph.
    print("[SUPERVISOR] No URLs found - routing to search_subgraph")
    return Command(goto="search_subgraph")

In [12]:
def dispatch_fetchers(state: ResearchState):
    """Returns a Send per URL so fetch_url workers run in parallel (dynamic spawning)."""
    urls = state.candidate_urls
    print(f"\n[DISPATCH_FETCHERS] Spawning {min(len(urls), 5)} fetch workers")
    return [Send("fetch_url", {"url": url}) for url in urls[:5]]  # cap at 5 concurrent fetches

## Creation of Subgraph
The search stage (Wikipedia + web, run in parallel, then merged) is its own graph, compiled once and reused as a single node -- `"search_subgraph"` -- inside the main graph.

In [13]:
def create_search_subgraph():
    """Creates and returns the search subgraph for parallel Wikipedia and web searches."""
    search_subgraph = StateGraph(ResearchState)

    search_subgraph.add_node("search_wikipedia", search_wikipedia, cache_policy=CachePolicy(ttl=300))
    search_subgraph.add_node("search_web", search_web, cache_policy=CachePolicy(ttl=300))
    search_subgraph.add_node("merge_results", merge_results, cache_policy=CachePolicy(ttl=300))

    # Parallel execution: both searches start from the entry point
    search_subgraph.set_entry_point("search_wikipedia")
    search_subgraph.set_entry_point("search_web")

    # Both searches flow to merge
    search_subgraph.add_edge("search_wikipedia", "merge_results")
    search_subgraph.add_edge("search_web", "merge_results")

    # Merge is the exit point
    search_subgraph.set_finish_point("merge_results")

    return search_subgraph


search_subgraph = create_search_subgraph().compile()

## Creation of main graph
All nodes register with `cache_policy`; `search_subgraph` is added like any other node. `supervisor` is the entry point and every worker routes back to it, except the two terminal nodes (`error_handler`, `summarize_results`) which go straight to `END`.

In [14]:
main_graph = StateGraph(ResearchState)

# Worker nodes
main_graph.add_node("prepare_query", prepare_query, cache_policy=CachePolicy(ttl=300))
main_graph.add_node("error_handler", error_handler, cache_policy=CachePolicy(ttl=300))
main_graph.add_node("summarize_results", summarize_results, cache_policy=CachePolicy(ttl=300))

# Supervisor node (key coordinator using Command)
main_graph.add_node("supervisor", supervisor, cache_policy=CachePolicy(ttl=300))

# Search subgraph as a single node
main_graph.add_node("search_subgraph", search_subgraph)

# fetch_url worker (spawned multiple times via Send)
main_graph.add_node("fetch_url", fetch_url, cache_policy=CachePolicy(ttl=300))

# Dispatcher node - sets fetching_initiated and spawns workers
main_graph.add_node("dispatcher", lambda state: {"fetching_initiated": True}, cache_policy=CachePolicy(ttl=300))

main_graph.add_edge(START, "supervisor")

# Worker nodes return to the supervisor for the next decision
main_graph.add_edge("prepare_query", "supervisor")
main_graph.add_edge("search_subgraph", "supervisor")
main_graph.add_edge("fetch_url", "supervisor")

# Terminal nodes go directly to END
main_graph.add_edge("error_handler", END)
main_graph.add_edge("summarize_results", END)

# Dispatcher uses a conditional edge to spawn Send workers
main_graph.add_conditional_edges("dispatcher", dispatch_fetchers)

In [15]:
app = main_graph.compile(
    checkpointer=MemorySaver(),  # state history & checkpointing
    cache=InMemoryCache(),       # backs the cache_policy on each node
)

## Execution of Graph

In [16]:
initial_state = {
    "query": "What is the difference between python and cobol?",
    "candidate_urls": [],
    "page_texts": [],
    "draft_summary": "",
    "final_report": "",
    "errors": [],
    "fetching_initiated": False,
}

config = {
    "configurable": {"thread_id": "research_agent_1"},
    "recursion_limit": 50,  # hard stop to prevent infinite looping
}

final_state = app.invoke(initial_state, config=config)

print("\nSources:")
for url in final_state["candidate_urls"]:
    print(f"  - {url}")
print(f"\n{final_state['final_report']}")

[SUPERVISOR] No URLs found - routing to search_subgraph


  merged 3 unique source(s)
[SUPERVISOR] 3 URLs found - routing to dispatcher

[DISPATCH_FETCHERS] Spawning 3 fetch workers


[SUPERVISOR] 6 page texts collected - routing to summarize_results

Sources:
  - https://en.wikipedia.org/wiki/Python_(programming_language)
  - https://en.wikipedia.org/wiki/Union_type
  - https://en.wikipedia.org/wiki/Compiler

[Skipped LLM call: set OPENAI_API_KEY to generate a real summary]


## Enabling checkpoints
Because we gave the run a `thread_id`, we can inspect its full history and even resume it from any earlier step -- this is Time Travel.

In [17]:
state_history = list(app.get_state_history(config={"configurable": {"thread_id": "research_agent_1"}}))
sorted_history = sorted(state_history, key=lambda x: x.metadata.get("step", -1))

for i, snapshot in enumerate(sorted_history):
    values = snapshot.values
    if values:
        print(f"Step {i}: URLs={len(values.get('candidate_urls', []))}  "
              f"Texts={len(values.get('page_texts', []))}  "
              f"Errors={len(values.get('errors', []))}")
        if values.get("final_report"):
            print(f"  Final report: {values['final_report'][:80]}...")

Step 0: URLs=0  Texts=0  Errors=0
Step 1: URLs=0  Texts=0  Errors=0
Step 2: URLs=0  Texts=0  Errors=0
Step 3: URLs=3  Texts=3  Errors=1
Step 4: URLs=3  Texts=3  Errors=1
Step 5: URLs=3  Texts=3  Errors=1
Step 6: URLs=3  Texts=6  Errors=1
Step 7: URLs=3  Texts=6  Errors=1
Step 8: URLs=3  Texts=6  Errors=1
  Final report: [Skipped LLM call: set OPENAI_API_KEY to generate a real summary]...


In [18]:
# Resume execution from an earlier checkpoint (step 4, or the last one if the run was shorter).
step_index = min(4, len(sorted_history) - 1)
checkpoint_id = sorted_history[step_index].config["configurable"]["checkpoint_id"]
config_resume = {"configurable": {"thread_id": "research_agent_1", "checkpoint_id": checkpoint_id}}

resumed_state = app.invoke(None, config=config_resume)
print("\nFinal Report (from resumed execution):")
print(resumed_state["final_report"])


Final Report (from resumed execution):
[Skipped LLM call: set OPENAI_API_KEY to generate a real summary]
